# Day 3 - Customer Segment Classification

In Day 2, KMeans clustering was used to generate customer segments based on lead characteristics.

In this notebook, a supervised learning model (Random Forest Classifier) is trained using the generated segment labels.

The objective is to predict the segment of a new incoming lead without running KMeans again.

The trained model will later be deployed through FastAPI and integrated with Brevo Marketing Automation.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [13]:
df = pd.read_csv(
    r"D:\Project\Brevo automation\data\processed\ml_dataset.csv"
)

df.head()

,EMAIL,REGISTER_DATE,COURSE_1,COURSE_2,COURSE_3,COURSE_COUNT,MULTI_COURSE,SOURCE_TYPE,SCHOLARSHIP,REGISTER_MONTH,REGISTER_DAY_OF_WEEK,SEGMENT
0,anhmol2k5@gmail.com,2026-06-30 13:07:00,Business Intelligence (BI),NaN,NaN,1,False,CLUB,False,6,Tuesday,BI_CLUB
1,tramanhnguyen1226@gmail.com,2026-06-30 13:07:00,Business Intelligence (BI),NaN,NaN,1,False,CLUB,False,6,Tuesday,BI_CLUB
2,n.duylong145@gmail.com,2026-06-30 13:07:00,Business Intelligence (BI),NaN,NaN,1,False,CLUB,False,6,Tuesday,BI_CLUB
3,hoangnama2k58@gmail.com,2026-06-30 13:07:00,Business Intelligence (BI),NaN,NaN,1,False,CLUB,False,6,Tuesday,BI_CLUB
4,Buidanghoangyen@gmail.com,2026-06-30 13:07:00,Business Intelligence (BI),NaN,NaN,1,False,CLUB,False,6,Tuesday,BI_CLUB


In [14]:
features = [
    "COURSE_1",
    "COURSE_2",
    "COURSE_3",
    "COURSE_COUNT",
    "MULTI_COURSE",
    "SOURCE_TYPE",
    "SCHOLARSHIP",
    "REGISTER_MONTH",
    "REGISTER_DAY_OF_WEEK"
]

X = df[features]

y = df["SEGMENT"]

In [15]:
print(X.dtypes)
print(y.value_counts())
X["COURSE_2"] = X["COURSE_2"].fillna("NONE")
X["COURSE_3"] = X["COURSE_3"].fillna("NONE")

COURSE_1                    str
COURSE_2                    str
COURSE_3                float64
COURSE_COUNT              int64
MULTI_COURSE               bool
SOURCE_TYPE                 str
SCHOLARSHIP                bool
REGISTER_MONTH            int64
REGISTER_DAY_OF_WEEK        str
dtype: object
SEGMENT
BI_CLUB             439
GENAI_UNIVERSITY    217
BI_UNIVERSITY        98
BA_SOCIAL            92
DA_OTHER             60
AI_AGENT_CLUB        33
COURSE_CLUB          17
BA_OTHER              9
DA_CLUB               3
Name: count, dtype: int64


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
categorical_features = [
    "COURSE_1",
    "COURSE_2",
    "COURSE_3",
    "SOURCE_TYPE",
    "REGISTER_DAY_OF_WEEK"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [18]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [19]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)
label_encoder.classes_

array(['AI_AGENT_CLUB', 'BA_OTHER', 'BA_SOCIAL', 'BI_CLUB',
       'BI_UNIVERSITY', 'COURSE_CLUB', 'DA_CLUB', 'DA_OTHER',
       'GENAI_UNIVERSITY'], dtype=object)

## Train-Test Split

The dataset is divided into training and testing sets.

- Training set: 80%
- Testing set: 20%

The testing set is used to evaluate how well the model generalizes to unseen data.

In [ ]:
X = pd.get_dummies(X, columns=["COURSE_1", "COURSE_2", "COURSE_3", "SOURCE_TYPE", "REGISTER_DAY_OF_WEEK", "SEGMENT"], drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (774, 9)
Testing samples: (194, 9)


## Random Forest Classifier

Random Forest is an ensemble learning algorithm that combines multiple decision trees.

Advantages:

- Handles categorical features after one-hot encoding.
- Robust to noise.
- Good performance with little parameter tuning.
- Suitable for multi-class classification.

In [21]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

ValueError: could not convert string to float: 'Generative AI (GenAI)'

In [19]:
y_pred = rf.predict(X_test)

In [20]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 1.0000


In [21]:
f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print(f"Weighted F1 Score: {f1:.4f}")

Weighted F1 Score: 1.0000


In [22]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

                     precision    recall  f1-score   support

        BA_FACEBOOK       1.00      1.00      1.00        20
          BI_AIESEC       1.00      1.00      1.00        37
BI_CLUB_SCHOLARSHIP       1.00      1.00      1.00        62
     BI_SCHOLARSHIP       1.00      1.00      1.00        20
     DA_SCHOLARSHIP       1.00      1.00      1.00        12
          GENAI_VNU       1.00      1.00      1.00        43

           accuracy                           1.00       194
          macro avg       1.00      1.00      1.00       194
       weighted avg       1.00      1.00      1.00       194



In [23]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

cm_df

,BA_FACEBOOK,BI_AIESEC,BI_CLUB_SCHOLARSHIP,BI_SCHOLARSHIP,DA_SCHOLARSHIP,GENAI_VNU
BA_FACEBOOK,20,0,0,0,0,0
BI_AIESEC,0,37,0,0,0,0
BI_CLUB_SCHOLARSHIP,0,0,62,0,0,0
BI_SCHOLARSHIP,0,0,0,20,0,0
DA_SCHOLARSHIP,0,0,0,0,12,0
GENAI_VNU,0,0,0,0,0,43


## Model Evaluation

The Random Forest classifier achieved perfect performance on the test set (Accuracy = 1.00, Weighted F1 = 1.00).

This result is expected because:

- The target labels (SEGMENT) were generated using KMeans based on the same input features.
- The dataset contains well-separated customer groups with highly distinctive characteristics.
- Therefore, the classifier successfully learned the mapping between lead attributes and the generated customer segments.

In a real-world application with more behavioral features and continuously changing customer data, the classification task would be significantly more challenging.

## Save Trained Model

The trained model and its supporting files are saved for deployment.

These files will later be loaded by the FastAPI service to predict the customer segment of new incoming leads.

In [24]:
joblib.dump(rf, "../models/random_forest.pkl")

print("Random Forest model saved.")

Random Forest model saved.


In [25]:
joblib.dump(
    label_encoder,
    "../models/label_encoder.pkl"
)

print("Label Encoder saved.")

Label Encoder saved.


In [26]:
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print("Feature columns saved.")

Feature columns saved.


In [27]:
model = joblib.load("../models/random_forest.pkl")

encoder = joblib.load("../models/label_encoder.pkl")

columns = joblib.load("../models/feature_columns.pkl")

print(type(model))
print(type(encoder))
print(len(columns))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.preprocessing._label.LabelEncoder'>
26
